In [1]:
print("kernel alive")


kernel alive


In [2]:
import os, re, json
from openai import OpenAI

# --- CONFIG ---
FILE_PATH = os.path.expanduser("~/novel-ai/prompts/examples/ch01_test.txt")
LLM_API_URL = "http://127.0.0.1:8080/v1"
LLM_API_KEY = "local"
MODEL = "qwen2.5-7b"

client = OpenAI(base_url=LLM_API_URL, api_key=LLM_API_KEY)

# --- LOAD TEXT ---
with open(FILE_PATH, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read().strip()

len(text), text[:300]


(16927,
 'Night draped over the city of Noctis.\n\nBeneath its hollow towers and rust-stained valleys, a memory stirred. Not a thought. Not a name. But a scent. Something fresh, new. The scent of minds not yet hardened - soft, porous, still becoming. It moved like a wolf in the fog. Slow. Silent. Chasing that s')

In [3]:
def split_paragraphs(t: str):
    # Tách theo đoạn trống (giữ nhịp văn tốt hơn tách theo câu)
    return [p.strip() for p in re.split(r"\n\s*\n", t) if p.strip()]

def build_chunks(paragraphs, max_chars=5200):
    # ~5.2k chars/chunk: chừa chỗ cho system+user prompt + JSON output trong 8k context
    chunks, cur, cur_len = [], [], 0
    for p in paragraphs:
        add_len = len(p) + 2
        if cur and (cur_len + add_len) > max_chars:
            chunks.append("\n\n".join(cur))
            cur, cur_len = [], 0
        cur.append(p)
        cur_len += add_len
    if cur:
        chunks.append("\n\n".join(cur))
    return chunks

paras = split_paragraphs(text)
chunks = build_chunks(paras, max_chars=5200)

print("paragraphs =", len(paras))
print("chunks     =", len(chunks))
print("chunk_sizes(first 5) =", [len(c) for c in chunks[:5]])
print("\n--- CHUNK 1 PREVIEW ---\n")
print(chunks[0][:800])


paragraphs = 159
chunks     = 4
chunk_sizes(first 5) = [4990, 5184, 5107, 1634]

--- CHUNK 1 PREVIEW ---

Night draped over the city of Noctis.

Beneath its hollow towers and rust-stained valleys, a memory stirred. Not a thought. Not a name. But a scent. Something fresh, new. The scent of minds not yet hardened - soft, porous, still becoming. It moved like a wolf in the fog. Slow. Silent. Chasing that scent.

Not every child. Not every dream. Only the ones that tasted right. That night, it came close. Close enough to brush past the pulse of three lives. Children slept in their mother’s arms. No one noticed when the space turned weightless, when the trees outside grew restless, as if stirred by a wind that didn’t blow.

And then, it passed. Quiet as it came. But something… had already seeped in.

…

Twelve years later.

Midday in the highlands carried a kind of sunlight unlike any other, not ha


In [5]:
SYSTEM = """You are The Analyst-Teacher for an offline grim sci-fi novel system.
Task: extract the author's style fingerprint and hidden logic (Chekhov's guns) from the given text.
Constraints:
- Do NOT continue the story.
- Do NOT rewrite large portions of text.
- Be technical, actionable, and consistent.
- Evidence quotes: max 20 words each.

Return ONLY valid JSON with this exact schema:
{
  "style_fingerprint": {
    "tone": "",
    "pov_camera": "",
    "pacing_profile": "",
    "sentence_rhythm": "",
    "imagery_motifs": [],
    "diction_rules": [],
    "dialogue_rules": [],
    "micro_style_rules": [],
    "taboo_list": []
  },
  "hidden_logic_checklist": [
    {"item":"", "type":"object|promise|threat|mystery|relationship|system", "status":"unresolved|seeded", "evidence_quote":"", "where":""}
  ],
  "prompt_snippets": {
    "system_prompt_patch_md": "",
    "editor_checklist": []
  }
}
"""

USER_TMPL = """Analyze excerpt chunk {i}/{n}.
Focus on:
1) style fingerprint (tone/POV/pacing/rhythm/diction/dialogue)
2) hidden logic checklist (Chekhov's guns)
3) produce concise rules that can be pasted into a system prompt later

TEXT:
{chunk}
"""

def llm_json(messages, temperature=0.3, max_tokens=1800):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content

# 1) phân tích từng chunk -> JSON
chunk_json_texts = []
for i, ch in enumerate(chunks, start=1):
    out = llm_json(
        [
            {"role":"system","content": SYSTEM},
            {"role":"user","content": USER_TMPL.format(i=i, n=len(chunks), chunk=ch)}
        ],
        temperature=0.3,
        max_tokens=1800
    )
    chunk_json_texts.append(out)
    print(f"chunk {i} done, chars={len(out)}")

# 2) merge các JSON chunk -> 1 JSON cuối
MERGE_SYSTEM = """You are a strict JSON merger.
Input: multiple JSON objects in the same schema.
Output: ONE merged JSON in the same schema.
Rules:
- Deduplicate lists (micro rules, motifs, checklist items) by semantic similarity.
- Keep the best, most specific rules.
- hidden_logic_checklist: keep unique items, prefer clearer evidence_quote.
Return ONLY valid JSON.
"""

merged = llm_json(
    [
        {"role":"system","content": MERGE_SYSTEM},
        {"role":"user","content": "Merge these JSON analyses:\n\n" + "\n\n---\n\n".join(chunk_json_texts)}
    ],
    temperature=0.2,
    max_tokens=2200
)

data = json.loads(merged)
print(json.dumps(data, indent=2, ensure_ascii=False)[:5000])


chunk 1 done, chars=2609
chunk 2 done, chars=2299
chunk 3 done, chars=2635
chunk 4 done, chars=2336


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [6]:
import json, re

print("merged preview:", repr(merged[:300]))

def extract_json(s: str):
    # 1) bóc từ code fence ```json ... ```
    m = re.search(r"```(?:json)?\s*(\{.*\})\s*```", s, flags=re.DOTALL)
    if m:
        return m.group(1).strip()
    # 2) bóc từ ký tự { đầu tiên đến } cuối cùng
    i = s.find("{")
    j = s.rfind("}")
    if i != -1 and j != -1 and j > i:
        return s[i:j+1].strip()
    return None

candidate = extract_json(merged)

if candidate is None:
    print("No JSON block found. Will ask LLM to repair.")
    REPAIR_SYSTEM = "You are a JSON repair tool. Return ONLY valid JSON. No extra words."
    REPAIR_USER = f"""Fix this into ONE valid JSON object (same schema as requested). Output ONLY JSON.

TEXT TO FIX:
{merged}
"""
    repaired = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content": REPAIR_SYSTEM},
            {"role":"user","content": REPAIR_USER}
        ],
        temperature=0.0,
        max_tokens=2200
    ).choices[0].message.content
    candidate = extract_json(repaired) or repaired.strip()

data = json.loads(candidate)
print("✅ JSON loaded. Top keys:", list(data.keys()))
print(json.dumps(data, indent=2, ensure_ascii=False)[:5000])


merged preview: '```json\n{\n  "style_fingerprint": {\n    "tone": "Mysterious, introspective, slightly unsettling and melancholic",\n    "pov_camera": "First-person limited, shifting to third-person limited, omniscient, switching between Mike and Kuro",\n    "pacing_profile": "Slow, building tension, slow-building, intr'
✅ JSON loaded. Top keys: ['style_fingerprint', 'hidden_logic_checklist', 'prompt_snippets']
{
  "style_fingerprint": {
    "tone": "Mysterious, introspective, slightly unsettling and melancholic",
    "pov_camera": "First-person limited, shifting to third-person limited, omniscient, switching between Mike and Kuro",
    "pacing_profile": "Slow, building tension, slow-building, introspective, with sudden shifts in mood, gradual, building tension through small details",
    "sentence_rhythm": "Varied, with long descriptive sentences interspersed with short, impactful ones, variable, with long descriptive sentences and shorter, more direct statements",
    "imagery_motif

In [1]:
CLEAN_SYSTEM = """You are a meticulous style auditor.
Given: the original text and a draft style fingerprint JSON.
Task: correct factual errors (especially POV), deduplicate, and rewrite rules to be crisp for a 7B model.
Return ONLY valid JSON (no markdown, no code fences). Same schema as input (style_fingerprint, hidden_logic_checklist, prompt_snippets)."""

CLEAN_USER = f"""ORIGINAL TEXT (for verification):
{text[:8000]}

DRAFT JSON (may contain errors):
{json.dumps(data, ensure_ascii=False)}

Fix errors by checking against the ORIGINAL TEXT. Especially:
- POV / camera perspective
- who is focalized
- what is actually present vs inferred

Output ONLY JSON.
"""

cleaned = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role":"system","content": CLEAN_SYSTEM},
        {"role":"user","content": CLEAN_USER}
    ],
    temperature=0.1,
    max_tokens=2200
).choices[0].message.content

clean_data = json.loads(cleaned)
print(json.dumps(clean_data["style_fingerprint"], indent=2, ensure_ascii=False))


NameError: name 'text' is not defined

In [3]:
# --- LOAD STYLE PATCH ---
STYLE_PATCH_PATH = os.path.expanduser("~/novel-ai/prompts/style_patch_v1_1.txt")
with open(STYLE_PATCH_PATH, "r", encoding="utf-8") as f:
    STYLE_PATCH = f.read().strip()

SUM_SYSTEM = f"""You are The Summarizer (Strict).
Use the STYLE PATCH below to keep tone/POV consistent.
Goal: compress content without losing hidden logic (Chekhov's guns).
Return ONLY valid JSON. No markdown, no code fences.

STYLE PATCH:
{STYLE_PATCH}

JSON SCHEMA:
{{
  "chapter_id": "",
  "summary_md": "",
  "key_events": [""],
  "character_state_changes": [
    {{"character":"","before":"","after":""}}
  ],
  "chekhov_checklist": [
    {{"item":"","type":"object|promise|threat|mystery|relationship|system","status":"seeded|unresolved","evidence_quote":"","where":""}}
  ],
  "questions_to_carry_forward": [""]
}}
"""

SUM_USER_TMPL = """Summarize the following text.
Rules:
- Preserve ambiguity; do NOT explain mysteries.
- Capture only what must be remembered to write later arcs.
- Keep summary concise but complete.

TEXT:
{chunk}
"""

def summarize_chunk(chunk):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content": SUM_SYSTEM},
            {"role":"user","content": SUM_USER_TMPL.format(chunk=chunk)}
        ],
        temperature=0.2,
        max_tokens=1400
    )
    return resp.choices[0].message.content

# Summarize all chunks
sum_chunks = []
for i, ch in enumerate(chunks, start=1):
    out = summarize_chunk(ch)
    sum_chunks.append(out)
    print(f"summary chunk {i} done")

# Merge summaries into one chapter-level JSON
MERGE_SUM_SYSTEM = "Merge multiple JSON summaries into ONE chapter-level JSON. Return ONLY JSON."
merged_sum = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role":"system","content": MERGE_SUM_SYSTEM},
        {"role":"user","content": "Merge these summaries:\n\n" + "\n\n---\n\n".join(sum_chunks)}
    ],
    temperature=0.1,
    max_tokens=1800
).choices[0].message.content

# Repair + load
import re, json
def extract_json(s):
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    return m.group(0) if m else s

chapter_memory = json.loads(extract_json(merged_sum))
print(json.dumps(chapter_memory, indent=2, ensure_ascii=False)[:5000])


NameError: name 'os' is not defined

In [ ]:
# --- LOAD STYLE PATCH ---
STYLE_PATCH_PATH = os.path.expanduser("~/novel-ai/prompts/style_patch_v1_1.txt")
with open(STYLE_PATCH_PATH, "r", encoding="utf-8") as f:
    STYLE_PATCH = f.read().strip()

SUM_SYSTEM = f"""You are The Summarizer (Strict).
Use the STYLE PATCH below to keep tone/POV consistent.
Goal: compress content without losing hidden logic (Chekhov's guns).
Return ONLY valid JSON. No markdown, no code fences.

STYLE PATCH:
{STYLE_PATCH}

JSON SCHEMA:
{{
  "chapter_id": "",
  "summary_md": "",
  "key_events": [""],
  "character_state_changes": [
    {{"character":"","before":"","after":""}}
  ],
  "chekhov_checklist": [
    {{"item":"","type":"object|promise|threat|mystery|relationship|system","status":"seeded|unresolved","evidence_quote":"","where":""}}
  ],
  "questions_to_carry_forward": [""]
}}
"""

SUM_USER_TMPL = """Summarize the following text.
Rules:
- Preserve ambiguity; do NOT explain mysteries.
- Capture only what must be remembered to write later arcs.
- Keep summary concise but complete.

TEXT:
{chunk}
"""

def summarize_chunk(chunk):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content": SUM_SYSTEM},
            {"role":"user","content": SUM_USER_TMPL.format(chunk=chunk)}
        ],
        temperature=0.2,
        max_tokens=1400
    )
    return resp.choices[0].message.content

# Summarize all chunks
sum_chunks = []
for i, ch in enumerate(chunks, start=1):
    out = summarize_chunk(ch)
    sum_chunks.append(out)
    print(f"summary chunk {i} done")

# Merge summaries into one chapter-level JSON
MERGE_SUM_SYSTEM = "Merge multiple JSON summaries into ONE chapter-level JSON. Return ONLY JSON."
merged_sum = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role":"system","content": MERGE_SUM_SYSTEM},
        {"role":"user","content": "Merge these summaries:\n\n" + "\n\n---\n\n".join(sum_chunks)}
    ],
    temperature=0.1,
    max_tokens=1800
).choices[0].message.content

# Repair + load
import re, json
def extract_json(s):
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    return m.group(0) if m else s

chapter_memory = json.loads(extract_json(merged_sum))
print(json.dumps(chapter_memory, indent=2, ensure_ascii=False)[:5000])


In [1]:
print("ping")


ping


In [3]:
# === BOOTSTRAP / MUST RUN FIRST ===
import os
import json
import re

from openai import OpenAI

# client for local llama.cpp server
client = OpenAI(
    base_url="http://127.0.0.1:8080/v1",
    api_key="local"
)

MODEL = "qwen2.5-7b"

print("bootstrap OK")


bootstrap OK


In [4]:
FILE_PATH = os.path.expanduser("~/novel-ai/prompts/examples/ch01_test.txt")
with open(FILE_PATH, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read().strip()

print("text chars =", len(text))
print(text[:120])


text chars = 16927
Night draped over the city of Noctis.

Beneath its hollow towers and rust-stained valleys, a memory stirred. Not a thoug


In [ ]:
def split_paragraphs(t: str):
    return [p.strip() for p in re.split(r"\n\s*\n", t) if p.strip()]

def build_chunks(paragraphs, max_chars=5200):
    chunks, cur, cur_len = [], [], 0
    for p in paragraphs:
        add_len = len(p) + 2
        if cur and (cur_len + add_len) > max_chars:
            chunks.append("\n\n".join(cur))
            cur, cur_len = [], 0
        cur.append(p)
        cur_len += add_len
    if cur:
        chunks.append("\n\n".join(cur))
    return chunks

paras = split_paragraphs(text)
chunks = build_chunks(paras, max_chars=5200)

print("paragraphs =", len(paras))
print("chunks =", len(chunks), "sizes =", [len(c) for c in chunks])


In [1]:
# --- LOAD STYLE PATCH ---
STYLE_PATCH_PATH = os.path.expanduser("~/novel-ai/prompts/style_patch_v1_1.txt")
with open(STYLE_PATCH_PATH, "r", encoding="utf-8") as f:
    STYLE_PATCH = f.read().strip()

SUM_SYSTEM = f"""You are The Summarizer (Strict).
Use the STYLE PATCH below to keep tone/POV consistent.
Goal: compress content without losing hidden logic (Chekhov's guns).
Return ONLY valid JSON. No markdown, no code fences.

STYLE PATCH:
{STYLE_PATCH}

JSON SCHEMA:
{{
  "chapter_id": "",
  "summary_md": "",
  "key_events": [""],
  "character_state_changes": [
    {{"character":"","before":"","after":""}}
  ],
  "chekhov_checklist": [
    {{"item":"","type":"object|promise|threat|mystery|relationship|system","status":"seeded|unresolved","evidence_quote":"","where":""}}
  ],
  "questions_to_carry_forward": [""]
}}
"""

SUM_USER_TMPL = """Summarize the following text.
Rules:
- Preserve ambiguity; do NOT explain mysteries.
- Capture only what must be remembered to write later arcs.
- Keep summary concise but complete.

TEXT:
{chunk}
"""

def summarize_chunk(chunk):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content": SUM_SYSTEM},
            {"role":"user","content": SUM_USER_TMPL.format(chunk=chunk)}
        ],
        temperature=0.2,
        max_tokens=1400
    )
    return resp.choices[0].message.content

# Summarize all chunks
sum_chunks = []
for i, ch in enumerate(chunks, start=1):
    out = summarize_chunk(ch)
    sum_chunks.append(out)
    print(f"summary chunk {i} done")

# Merge summaries into one chapter-level JSON
MERGE_SUM_SYSTEM = "Merge multiple JSON summaries into ONE chapter-level JSON. Return ONLY JSON."
merged_sum = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role":"system","content": MERGE_SUM_SYSTEM},
        {"role":"user","content": "Merge these summaries:\n\n" + "\n\n---\n\n".join(sum_chunks)}
    ],
    temperature=0.1,
    max_tokens=1800
).choices[0].message.content

# Repair + load
import re, json
def extract_json(s):
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    return m.group(0) if m else s

chapter_memory = json.loads(extract_json(merged_sum))
print(json.dumps(chapter_memory, indent=2, ensure_ascii=False)[:5000])


NameError: name 'os' is not defined

In [2]:
print("ping")

ping


In [1]:
import os
import json
import re
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8080/v1",
    api_key="local"
)

MODEL = "qwen2.5-7b"

print("bootstrap OK")


bootstrap OK


In [2]:
FILE_PATH = os.path.expanduser("~/novel-ai/prompts/examples/ch01_test.txt")
with open(FILE_PATH, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read().strip()

print("text chars =", len(text))


text chars = 16927


In [3]:
def split_paragraphs(t: str):
    return [p.strip() for p in re.split(r"\n\s*\n", t) if p.strip()]

def build_chunks(paragraphs, max_chars=5200):
    chunks, cur, cur_len = [], [], 0
    for p in paragraphs:
        add_len = len(p) + 2
        if cur and (cur_len + add_len) > max_chars:
            chunks.append("\n\n".join(cur))
            cur, cur_len = [], 0
        cur.append(p)
        cur_len += add_len
    if cur:
        chunks.append("\n\n".join(cur))
    return chunks

paras = split_paragraphs(text)
chunks = build_chunks(paras)

print("chunks =", len(chunks))


chunks = 4


In [4]:
STYLE_PATCH_PATH = os.path.expanduser("~/novel-ai/prompts/style_patch_v1_1.txt")
with open(STYLE_PATCH_PATH, "r", encoding="utf-8") as f:
    STYLE_PATCH = f.read().strip()

print("style patch chars =", len(STYLE_PATCH))


style patch chars = 1925


In [5]:
# === SUMMARIZER v0 (chapter memory) ===

SUM_SYSTEM = f"""You are The Summarizer (Strict).
Use the STYLE PATCH below to keep tone/POV consistent.
Goal: compress content without losing hidden logic (Chekhov's guns).
Return ONLY valid JSON. No markdown, no code fences.

STYLE PATCH:
{STYLE_PATCH}

JSON SCHEMA:
{{
  "chapter_id": "ch01_test",
  "summary_md": "",
  "key_events": [""],
  "character_state_changes": [
    {{"character":"","before":"","after":""}}
  ],
  "chekhov_checklist": [
    {{"item":"","type":"object|promise|threat|mystery|relationship|system","status":"seeded|unresolved","evidence_quote":"","where":""}}
  ],
  "questions_to_carry_forward": [""]
}}
"""

SUM_USER_TMPL = """Summarize the following text.
Rules:
- Preserve ambiguity; do NOT explain mysteries.
- Capture only what must be remembered to write later arcs.
- Keep summary concise but complete.

TEXT:
{chunk}
"""

def extract_json_block(s: str):
    # try codefence first
    m = re.search(r"```(?:json)?\s*(\{.*\})\s*```", s, flags=re.DOTALL)
    if m:
        return m.group(1).strip()
    # then first { ... last }
    i, j = s.find("{"), s.rfind("}")
    if i != -1 and j != -1 and j > i:
        return s[i:j+1].strip()
    return s.strip()

def llm_call(system, user, temperature=0.2, max_tokens=1400):
    return client.chat.completions.create(
        model=MODEL,
        messages=[{"role":"system","content":system},{"role":"user","content":user}],
        temperature=temperature,
        max_tokens=max_tokens
    ).choices[0].message.content

def summarize_chunk(chunk, idx):
    out = llm_call(SUM_SYSTEM, SUM_USER_TMPL.format(chunk=chunk), temperature=0.2, max_tokens=1400)
    try:
        return json.loads(extract_json_block(out))
    except Exception:
        # repair pass
        repaired = llm_call(
            "You are a JSON repair tool. Return ONLY valid JSON.",
            "Fix into valid JSON following the schema. Output ONLY JSON.\n\nTEXT:\n" + out,
            temperature=0.0,
            max_tokens=1400
        )
        return json.loads(extract_json_block(repaired))

# 1) summarize each chunk -> dict
sum_dicts = []
for i, ch in enumerate(chunks, start=1):
    print(f"[summarize] chunk {i}/{len(chunks)} ...")
    d = summarize_chunk(ch, i)
    sum_dicts.append(d)

# 2) merge chunk summaries -> one chapter memory
merge_in = json.dumps(sum_dicts, ensure_ascii=False)
merged_out = llm_call(
    "Merge multiple chunk-level JSON summaries into ONE chapter-level JSON using the schema. Return ONLY JSON.",
    merge_in,
    temperature=0.1,
    max_tokens=1800
)

chapter_memory = json.loads(extract_json_block(merged_out))

print("\n✅ chapter_memory keys:", list(chapter_memory.keys()))
print(json.dumps(chapter_memory, indent=2, ensure_ascii=False)[:5000])


[summarize] chunk 1/4 ...
[summarize] chunk 2/4 ...
[summarize] chunk 3/4 ...
[summarize] chunk 4/4 ...

✅ chapter_memory keys: ['chapter_id', 'summary_md', 'key_events', 'character_state_changes', 'chekhov_checklist', 'questions_to_carry_forward']
{
  "chapter_id": "ch01_test",
  "summary_md": "",
  "key_events": [
    "A mysterious presence stirs in Noctis, sensing fresh, soft minds.",
    "Kuro Sora, a fourteen-year-old from Firel, lives in Noctis, feeling distant and uninterested in his peers.",
    "Mike Aeran, a tech-savvy classmate, notices Kuro's detachment and suggests a path in tech.",
    "Kuro contemplates a future in software engineering but desires freedom from his father's path.",
    "Kuro fakes a stomach ache to leave class early and walks to the basalt fields where he feels a strange sensation.",
    "Mike warns Kuro about being noticed and mentions feeling something weird after class.",
    "Kuro dreams of a silent path with a shape at the end, feeling unsettled by a